In [ ]:
import os
import sys
sys.path.append('/dataset/')

import math
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from classifier import MLP
from dataset.MNIST import MNIST
from dataset.FMNIST import FMNIST
from dataset.CIFAR10 import CIFAR10
# from torchvision.datasets import MNIST
from dataset.MNISTPerClass import MNISTPerClass
from dataset.FMNISTPerClass import FMNISTPerClass
from dataset.CIFAR10PerClass import CIFARPerClass
from Autoencoder_functions import koopman_loss, collect_latent_states
from torch.nn.utils import parameters_to_vector
from scipy.linalg import eig, inv
from torch.utils.tensorboard import SummaryWriter
from torch.nn.utils.stateless import functional_call
from tensorboard import notebook

from tqdm import tqdm


Parameters

In [ ]:
kae_model_num = 0 # 0: nominal | 1: deeper
if kae_model_num == 0:
    from Autoencoder_real import KoopmanAutoencoder
elif kae_model_num == 1:
    from Autoencoder_real_v2 import KoopmanAutoencoder

# Use GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cuda:1')
print('Currently using... '+str(device))

combinations = [
    (hk, kae, sub, eig, clf, num_m, td) 
    for hk in [11] # hidden_k
    for kae in [3] # loss_kae weight
    for sub in [-1] # loss_sub weight (FIX AS -1)
    for eig in [-1] # loss_eig weight (FIX AS -1)
    for clf in [3] # loss_kae_classifier weight
    for num_m in [-1] # number of dominant mode (FIX AS -1)
    for td in ['CIFAR10']
    
]

lr_kae = 1e-5 
num_epochs = 2


Functions

In [ ]:
def compute_gradient_norm(model, norm_type=2):
    with torch.no_grad():
        total_norm = 0.0
        for param in model.parameters():
            if param.grad is not None:
                param_norm = param.grad.norm(norm_type)
                total_norm += param_norm.item() ** norm_type
        total_norm = total_norm ** (1.0 / norm_type)
    return total_norm

def classifier_sub(x, p_vec):
    idx, p_recon = 0, []
    for layer in classifier_shapes:
        layer_params = []
        for shape in layer:
            offset = np.prod(shape)
            layer_params.append(p_vec[idx:idx+offset].reshape(shape))
            idx += offset
        p_recon.append(layer_params)
    
    w0, b0 = p_recon[0]
    w1, b1 = p_recon[1]
    x = F.linear(x, w0, b0)
    x = F.relu(x)
    x = F.linear(x, w1, b1)
    return x

def compute_l_classifier(model, images, labels):
    # Move tensors to device
    images = images.reshape(-1, 28*28).to(device)
    labels = labels.to(device)

    # Forward pass
    outputs = model(images)
    loss = criterion_classifier(outputs, labels)
    return loss

def compute_l_kae(kae, params_snapshots):
    x = torch.stack(params_snapshots, dim=0).to(device)
    latents, latents_next = collect_latent_states(kae, x)
    kae.compute_koopman_operator(latents, latents_next)
    x_hat, z, z_pred = kae(x)
    recon_loss, state_pred_loss, koopman_pred_loss = koopman_loss(x, x_hat, z_pred, p, kae)
    loss_kae = c1*recon_loss + c2*state_pred_loss + c3*koopman_pred_loss # + c4*k_norm_loss      
    return loss_kae, z


def compute_theta_sub_all(kae, z, ko):
    # ko = kae.K.detach()
    eigvals, eigvec_left = torch.linalg.eig(ko)
    eigvec_left = eigvec_left.real.detach()
    # for e in range(hidden_k):
        # writer.add_scalar(f'Eigval/{e}', torch.abs(eigvals[e]), n_batch * epoch + inner)
    # B = np.pad(np.eye(n_params), ((0, 0), (0, N_O - n_params)), mode='constant')
    eigvec_left_inv = torch.linalg.pinv(eigvec_left)
    v = (kae.decoder(eigvec_left_inv)).T
    phi = eigvec_left @ z[-1, :]
    param_sub_all = v @ torch.diag(phi)
    return param_sub_all, eigvals

# def compute_theta_sub_all_steps(kae, z, ko, n):
#     # ko = kae.K
#     ko = torch.linalg.matrix_power(ko,n)
#     eigvals, eigvec_left = torch.linalg.eig(ko)
#     eigvec_left = eigvec_left.real.detach()
#     # for e in range(hidden_k):
#     #     writer.add_scalar(f'Eigval/{e}', torch.abs(eigvals[e]), n_batch * epoch + inner)
#     # B = np.pad(np.eye(n_params), ((0, 0), (0, N_O - n_params)), mode='constant')
#     eigvec_left_inv = torch.linalg.pinv(eigvec_left)
#     v = (kae.decoder(eigvec_left_inv)).T
#     phi = eigvec_left @ z[-1, :]
#     param_sub_all = v @ torch.diag(phi)
#     return param_sub_all, eigvals

# def compute_l_sub(param_sub, target_classes, images, labels):
#     # classifier_sub = MLP(image_size, hidden_c, num_classes).to(device)
#     # nn.utils.vector_to_parameters(param_sub, classifier_sub.parameters())
#     images = images.reshape(-1, 28*28).to(device)
#     labels = labels.to(device)
#     mask = torch.isin(labels, target_classes.clone().detach())
#     images = images[mask]
#     labels = labels[mask]
#     outputs = classifier_sub(images, param_sub)
#     loss = criterion_classifier(outputs, labels)
#     return loss

def compute_l_classifier_within(param_sub, images, labels):
    # classifier_sub = MLP(image_size, hidden_c, num_classes).to(device)
    # nn.utils.vector_to_parameters(param_sub, classifier_sub.parameters())
    images = images.reshape(-1, 28*28).to(device)
    labels = labels.to(device)
    outputs = classifier_sub(images, param_sub)
    loss = criterion_classifier(outputs, labels)
    return loss

def test_classifier(model, test_loader):
    model.eval()  # evaluation mode
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in test_loader:
            images = images.reshape(-1, 28*28).to(device)
            labels = labels.to(device)
            outputs = model(images)
            predicted = torch.argmax(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f'Test Accuracy: {accuracy:.2f}%')

# def get_target_classes(param_sub, candidates, images, labels):
#     with torch.autograd.no_grad():
#         images = images.reshape(-1, 28*28).to(device)
#         labels = labels.to(device)
#         outputs = classifier_sub(images, param_sub)
#         _, pred = torch.max(outputs, 1)
#         acc = []
#         for i in candidates:
#             if i != -1:
#                 mask = labels == i
#                 acc.append((pred[mask] == labels[mask]).sum()/mask.sum())
#             else:
#                 acc.append(-1)
#         _, top_idx = torch.topk(torch.tensor(acc), num_class_per_mode)
#         return candidates[top_idx]
    

In [ ]:
if __name__=='__main__':
    # Set training to be deterministic
    seed = 10
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

    # Fixed parameters (do not change)
    batch_size = 128 # 128
    lr_classifier = 1e-3 # 1e-5
    T = 5 
    p = 5
    # max_param_stack = 2**9 #batch_size - p - 1
    max_param_threshold = 0.15
    kae_break_threshold = 0.005
    c1, c2, c3 = 1, 1, 1
    image_size = 784  # 28x28 images flattened
    num_classes = 10

    # Varialbe initialization (do not change)
    stack_param = True
    learn_kae = False
    save = True
    
    ##################################################
    ############# PARAM SEARCH LOOP START ############
    ##################################################

    for hidden_k, kae_coef, _, _, kae_classifier_coef, _, target_dataset in combinations:
    ##########################################
        try: 
            # Info = [kae_coef, -1, -1, kae_classifier_coef, hidden_k, -1, target_dataset]
            Info = [kae_coef, -1, -1, kae_classifier_coef, hidden_k, -1]
            # num_class_per_mode = int(math.ceil(num_classes/num_mode_dom))
            for i in range(len(Info)):
                Info[i] = int(Info[i])

            print('kae_coef, sub_coef, eig_coef, kae_classifier_coef, hidden_k, num_mode_dom')
            print(Info)

            # Load datasets
            if target_dataset == 'MNIST':
                dataset_in_use_per_class = MNISTPerClass(batch_size=batch_size)
                dataset_in_use = MNIST(batch_size=batch_size)
                max_param_stack = 2 ** 9
                hidden_c = 16

            elif target_dataset == 'FMNIST':
                dataset_in_use_per_class = FMNISTPerClass(batch_size=batch_size)
                dataset_in_use = FMNIST(batch_size=batch_size)
                max_param_stack = 2 ** 9
                hidden_c = 16

            elif target_dataset == 'CIFAR10':
                dataset_in_use_per_class = CIFARPerClass(batch_size=batch_size)
                dataset_in_use = CIFAR10(batch_size=batch_size)
                max_param_stack = 2 ** 9
                hidden_c = 16

            # Define KAE test classifier
            kae_classifier = MLP(image_size, hidden_c, num_classes).to(device)
            kae_classifier.eval()

            # Build or load the classifier
            if os.path.isfile('results/classifier_'+str(target_dataset)+'_'+str(int(max_param_stack))+'.pth'):
                classifier = torch.load('results/classifier_'+str(target_dataset)+'_'+str(int(max_param_stack))+'.pth', weights_only=False, map_location=device)
                stack_param = False
                learn_kae = True
                print('load '+str(target_dataset)+' classifier')
            # elif os.path.isfile('results/classifier_FMNIST_'+str(int(max_param_stack))+'.pth') and target_dataset == 'FMNIST':
            #     classifier = torch.load('results/classifier_FMNIST_'+str(int(max_param_stack))+'.pth', weights_only=False, map_location=device)
            #     stack_param = False
            #     learn_kae = True
            #     print('load FMNIST classifier')
            # elif os.path.isfile('results/classifier_CIFAR10_'+str(int(max_param_stack))+'.pth') and target_dataset == 'CIFAR10':
            #     classifier = torch.load('results/classifier_CIFAR10_'+str(int(max_param_stack))+'.pth', weights_only=False, map_location=device)
            #     stack_param = False
            #     learn_kae = True
            #     print('load CIFAR10 classifier')
            #     # print('Load original classifier...')
            else:
                classifier = MLP(image_size, hidden_c, num_classes).to(device)
                classifier.train()
                print('No original classifier... start learning '+str(target_dataset)+' classifier')
            criterion_classifier = nn.CrossEntropyLoss()
            optimizer_classifier = optim.Adam(classifier.parameters(), lr=lr_classifier)
            classifier_shapes = [
                [(hidden_c, image_size), (hidden_c)],
                [(num_classes, hidden_c), (num_classes)]
            ]
            
            # Build the KAE
            param_vec = parameters_to_vector(classifier.parameters()) 
            state_dim = param_vec.shape[0]    
            kae = KoopmanAutoencoder(state_dim=state_dim, hidden_dim=hidden_k).to(device)
            kae.train()
            criterion_kae = koopman_loss
            mse = torch.nn.MSELoss()
            optimizer_kae = optim.Adam(kae.parameters(), lr=lr_kae)
            kae.train()
            
            # Get T number of snapshots first / Or load parameter history
            train_loader_classifier = dataset_in_use.train_loader
            test_loader = dataset_in_use.test_loader
            if stack_param:
                params_snapshots = []
                for epoch in tqdm(range(T)):
                    running_loss = 0.0
                    params_snapshots.append(parameters_to_vector(classifier.parameters()))
                    for images, labels in train_loader_classifier:
                        loss_classifier = compute_l_classifier(classifier, images, labels)
                        optimizer_classifier.zero_grad()
                        loss_classifier.backward()
                        optimizer_classifier.step()
                        running_loss += loss_classifier.item()
                test_classifier(classifier, test_loader)
                # print('Current classifier loss: ' + str(running_loss/len(train_loader_classifier)))
            else:
                temp=torch.load('results/params_snapshots_'+str(target_dataset)+'_'+str(int(max_param_stack))+'.pth', weights_only=False, map_location=device)
                params_snapshots = temp['params_snapshots']
            # elif target_dataset == 'FMNIST':
            #     temp=torch.load('results/params_snapshots_FMNIST_'+str(int(max_param_stack))+'.pth', weights_only=False, map_location=device)
            #     params_snapshots = temp['params_snapshots']
            # elif target_dataset == 'CIFAR10':
            #     temp=torch.load('results/params_snapshots_CIFAR10_'+str(int(max_param_stack))+'.pth', weights_only=False, map_location=device)
            #     params_snapshots = temp['params_snapshots']
            n_params = len(params_snapshots[0])
            classifier.train()
            n_batch = len(train_loader_classifier)

            try:
                kae = torch.load('results/kae_'+str(target_dataset)+'_'+str(Info)+'.pth', weights_only=False, map_location=device)
                print('model loaded, continue learning')
            except:
                try:
                    kae = torch.load('results/kae_'+str(target_dataset)+'_'+str(Info)+'_continue.pth', weights_only=False, map_location=device)
                    print('model loaded, continue learning')
                except:
                    print('no existing model')
            
            for epoch in tqdm(range(num_epochs)):
                running_loss = 0.0
                running_classifier_loss = 0.0
                # current_params = parameters_to_vector(classifier.parameters())
                for inner, (images, labels) in enumerate(train_loader_classifier):
                    # loss_sub = 0.0
                    # loss_eig = 0.0
                    loss_kae_classifier = 0.0
                    lose_kae = 0.0

                    if learn_kae:

                        # loss_kae + eigvals
                        loss_kae, z = compute_l_kae(kae, params_snapshots) # Koopman operator is updated here.
                        N_O = z.shape[-1]
                        param_sub_all, eigvals = compute_theta_sub_all(kae, z, kae.K)

                        order = torch.argsort(eigvals.abs())

                        # # loss_sub
                        # if sub_coef > 0:
                        #     candidates = torch.linspace(0, 9, 10, dtype=int, device=device)
                        #     for _ in range(num_mode_dom):
                        #         best_mode = torch.argmax(order)
                        #         order[best_mode] = -1
                        #         target_classes = get_target_classes(param_sub_all[:, best_mode], candidates, images, labels)
                        #         if -1 in candidates[target_classes]:
                        #             rest = ~(candidates == -1)
                        #             target_classes = candidates[rest]
                        #         candidates[target_classes] = -1

                        #         loss_sub = loss_sub + compute_l_sub(param_sub_all[:, best_mode], target_classes, images, labels)
                            
                        # # loss_eig
                        # if eig_coef > 0:
                        #     eig_target = torch.ones(num_mode_dom, device=device)
                        #     eig_compare = torch.zeros(num_mode_dom, device=device)
                        #     for ii in range(0,num_mode_dom):
                        #         temp = eigvals[ii]
                        #         eig_compare[ii] = torch.linalg.norm(temp - eig_target[ii])
                        #     loss_eig = torch.sum(eig_compare)

                        # KAE classifier loss
                        if kae_classifier_coef > 0:
                            for i in range(0,hidden_k):
                                if i == 0:
                                    param_sub_kae = param_sub_all[:, 0]
                                else:
                                    param_sub_kae = param_sub_kae + param_sub_all[:, i]
                                    
                            loss_kae_classifier = compute_l_classifier_within(param_sub_kae.to(device), images, labels)

                        # Total loss
                        # loss = kae_coef * loss_kae + sub_coef * loss_sub + eig_coef*loss_eig + kae_classifier_coef * loss_kae_classifier
                        loss = kae_coef * loss_kae + kae_classifier_coef * loss_kae_classifier
                        
                        optimizer_kae.zero_grad()
                        loss.backward()
                        optimizer_kae.step()
                        running_loss += loss.item()

                    if stack_param: # stack parameter history
                        loss_classifier = compute_l_classifier(classifier, images, labels)
                        params_snapshots.append(parameters_to_vector(classifier.parameters()))
                        optimizer_classifier.zero_grad()
                        loss_classifier.backward() 
                        optimizer_classifier.step()
                        running_classifier_loss += loss_classifier.item()

                    if (len(params_snapshots) >= max_param_stack) and stack_param: # stop stacking after max number of history
                        stack_param = False
                        print('parameter stack stopped with '+str(len(params_snapshots))+' steps history')
                        print('last loss_classifier = ' + str(loss_classifier))
                        test_classifier(classifier, test_loader)
                        torch.save(classifier, 'results/classifier_'+str(target_dataset)+'_'+str(int(max_param_stack))+'.pth')
                        torch.save({'params_snapshots': params_snapshots}, 'results/params_snapshots_'+str(target_dataset)+'_'+str(int(max_param_stack))+'.pth')  
                        learn_kae = True
                
                if save and epoch%50 == 0 and epoch > 1: # save waypoint & temporal model
                # if save and epoch > 0:
                    torch.save({
                    'epoch': epoch,
                    'params_sub': param_sub_all,
                    'param_original': classifier.state_dict(),
                    'param_kae': kae.state_dict(),
                    'optimizer_state_dict': optimizer_kae.state_dict(),
                    'ko': kae.K,
                    'Info': Info
                    }, 'waypoints/waypoint_'+str(target_dataset)+'_'+str(Info)+'.pth')
                    torch.save(kae, 'waypoints/kae_'+str(target_dataset)+'_'+str(Info)+'.pth')
                    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader_classifier):.4f}')


                if running_loss/len(train_loader_classifier) < kae_break_threshold and learn_kae:
                    break # break if converged
                elif math.isnan(running_loss/len(train_loader_classifier)):
                    print('NAN detected')
                    break

            torch.save(kae, 'results/kae_'+str(target_dataset)+'_'+str(Info)+'.pth') # save final model
            print(f'Iteration finished, loss = {running_loss/len(train_loader_classifier):.4f}')

        except Exception as e:
            print('ERROR OCCURED, SKIP CURRENT LOOP')
            print(e)
            pass
